In [1]:
!pip install pypdf

In [3]:
"""
Document Loader and Chunker for the AASTU Academic Handbook.
Reads PDF files and splits them into clean, overlapping text chunks with metadata.
"""

import os
from typing import List, Dict, Any
from pypdf import PdfReader


def load_pdf_pages(pdf_path: str) -> List[Dict[str, Any]]:
    """
    Reads a PDF file and extracts text page by page.

    Args:
        pdf_path: Absolute or relative path to the PDF file.

    Returns:
        A list of dictionaries, where each item contains:
        - page_number: The 1-based page index.
        - text: The raw text extracted from that page.
    """
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"PDF file not found at: {pdf_path}")

    reader = PdfReader(pdf_path)
    pages_data = []

    for idx, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        # Clean up excessive whitespace and line breaks
        cleaned_text = " ".join(text.split())
        if cleaned_text:
            pages_data.append({
                "page_number": idx + 1,
                "text": cleaned_text
            })

    return pages_data


def chunk_text(
    pages_data: List[Dict[str, Any]],
    chunk_size: int = 600,
    chunk_overlap: int = 100
) -> List[Dict[str, Any]]:
    """
    Splits page text into smaller overlapping chunks with metadata.

    Args:
        pages_data: List of page dicts from load_pdf_pages.
        chunk_size: Maximum characters per chunk (approx. 100-150 words).
        chunk_overlap: Number of characters shared between consecutive chunks.

    Returns:
        List of chunks with metadata: text, page_number, chunk_id, and source.
    """
    chunks = []
    chunk_counter = 0

    for page in pages_data:
        text = page["text"]
        page_num = page["page_number"]
        start = 0

        # Slide a window across the page text
        while start < len(text):
            end = start + chunk_size
            chunk_slice = text[start:end].strip()

            if chunk_slice:
                chunk_counter += 1
                chunks.append({
                    "chunk_id": f"chunk_{chunk_counter}",
                    "page_number": page_num,
                    "source": "AASTU Student Academic Handbook",
                    "text": chunk_slice
                })

            # Move forward by (chunk_size - chunk_overlap)
            start += (chunk_size - chunk_overlap)

    return chunks


def find_handbook_path() -> str:
    """Helper to locate the handbook PDF both locally and in Google Colab."""
    candidates = [
        os.path.join("Data", "AASTU Student Academic Handbook.pdf"),
        os.path.join("..", "Data", "AASTU Student Academic Handbook.pdf"),
        "/content/AASTU Student Academic Handbook.pdf",
        "AASTU Student Academic Handbook.pdf",
    ]
    for path in candidates:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(
        "Could not find 'AASTU Student Academic Handbook.pdf' in Data/ or current directory."
    )


if __name__ == "__main__":
    # Self-test block: runs when executed directly
    pdf_path = find_handbook_path()
    print(f"Testing document loader with: {pdf_path}")

    pages = load_pdf_pages(pdf_path)
    print(f"[OK] Successfully loaded {len(pages)} pages.")

    all_chunks = chunk_text(pages, chunk_size=600, chunk_overlap=100)
    print(f"[OK] Generated {len(all_chunks)} searchable chunks.")

    if all_chunks:
        first = all_chunks[0]
        print("\n--- Example First Chunk ---")
        print(f"ID: {first['chunk_id']} | Page: {first['page_number']}")
        print(f"Text preview: {first['text'][:200]}...")

Testing document loader with: /content/AASTU Student Academic Handbook.pdf
[OK] Successfully loaded 377 pages.
[OK] Generated 1465 searchable chunks.

--- Example First Chunk ---
ID: chunk_1 | Page: 1
Text preview: Student Handbook 2023/2024...


In [4]:
"""
Vector Store and Semantic Search Engine.
Converts text chunks into embeddings via Gemini API, computes cosine similarity,
and caches vectors locally for fast, repeatable retrieval.
"""

import os
import math
import json
import time
import urllib.request
import urllib.error
from typing import List, Dict, Any


def get_gemini_api_key() -> str:
    """
    Safely retrieves the Gemini API key from Google Colab Secrets
    or local .env / environment variables.
    """
    # 1. Try Google Colab userdata
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key:
            return key
    except Exception:
        pass

    # 2. Try python-dotenv for local development
    try:
        from dotenv import load_dotenv
        # Try current dir and parent dir for .env
        load_dotenv()
        load_dotenv(os.path.join(os.path.dirname(__file__), "..", ".env"))
    except Exception:
        pass

    key = os.getenv("GEMINI_API_KEY")
    if not key or key.startswith("your_"):
        raise ValueError(
            "GEMINI_API_KEY not found! Please set it in your .env file or Colab Secrets (🔑)."
        )
    return key


def cosine_similarity(v1: List[float], v2: List[float]) -> float:
    """Computes the cosine similarity between two numeric vectors."""
    dot_product = sum(a * b for a, b in zip(v1, v2))
    norm_v1 = math.sqrt(sum(a * a for a in v1))
    norm_v2 = math.sqrt(sum(b * b for b in v2))
    if norm_v1 == 0.0 or norm_v2 == 0.0:
        return 0.0
    return dot_product / (norm_v1 * norm_v2)


class SimpleVectorStore:
    """A lightweight vector store with batch embedding and disk caching."""

    def __init__(self, api_key: str = None):
        self.api_key = api_key or get_gemini_api_key()
        self.entries: List[Dict[str, Any]] = []

    def get_embedding(self, text: str) -> List[float]:
        """Embeds a single query string using gemini-embedding-001."""
        url = (
            f"https://generativelanguage.googleapis.com/v1beta/models/"
            f"gemini-embedding-001:embedContent?key={self.api_key}"
        )
        payload = {
            "model": "models/gemini-embedding-001",
            "content": {"parts": [{"text": text[:2000]}]}
        }
        data = json.dumps(payload).encode("utf-8")
        req = urllib.request.Request(url, data=data, headers={"Content-Type": "application/json"})

        with urllib.request.urlopen(req) as resp:
            result = json.loads(resp.read().decode("utf-8"))
            return result["embedding"]["values"]

    def add_chunks(self, chunks: List[Dict[str, Any]], batch_size: int = 20) -> None:
        """
        Embeds multiple text chunks using batchEmbedContents and stores them.
        """
        url = (
            f"https://generativelanguage.googleapis.com/v1beta/models/"
            f"gemini-embedding-001:batchEmbedContents?key={self.api_key}"
        )

        total = len(chunks)
        print(f"Embedding {total} chunks in batches of {batch_size}...")

        for i in range(0, total, batch_size):
            batch = chunks[i:i + batch_size]
            requests = [
                {
                    "model": "models/gemini-embedding-001",
                    "content": {"parts": [{"text": c["text"][:2000]}]}
                }
                for c in batch
            ]
            data = json.dumps({"requests": requests}).encode("utf-8")
            req = urllib.request.Request(url, data=data, headers={"Content-Type": "application/json"})

            try:
                with urllib.request.urlopen(req) as resp:
                    res = json.loads(resp.read().decode("utf-8"))
                    embeddings = res.get("embeddings", [])
                    for chunk, emb in zip(batch, embeddings):
                        self.entries.append({
                            "chunk_id": chunk["chunk_id"],
                            "page_number": chunk["page_number"],
                            "source": chunk.get("source", "AASTU Handbook"),
                            "text": chunk["text"],
                            "vector": emb["values"]
                        })
                print(f"  Processed {min(i + batch_size, total)}/{total} chunks...")
                time.sleep(0.5)  # Respect free API rate limits
            except urllib.error.HTTPError as e:
                print(f"  [Warning] HTTP Error {e.code}: {e.reason}. Retrying batch...")
                time.sleep(2)

    def save_cache(self, file_path: str) -> None:
        """Saves stored vectors and metadata to disk."""
        os.makedirs(os.path.dirname(os.path.abspath(file_path)), exist_ok=True)
        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(self.entries, f)
        print(f"[OK] Cached {len(self.entries)} vector entries to: {file_path}")

    def load_cache(self, file_path: str) -> bool:
        """Loads vectors from disk cache if the file exists."""
        if os.path.exists(file_path):
            with open(file_path, "r", encoding="utf-8") as f:
                self.entries = json.load(f)
            print(f"[OK] Loaded {len(self.entries)} vector entries from cache: {file_path}")
            return True
        return False

    def search(self, query: str, top_k: int = 3) -> List[Dict[str, Any]]:
        """
        Searches the stored chunks and returns the top-k most semantically relevant.
        """
        if not self.entries:
            raise ValueError("Vector store is empty! Add chunks or load from cache first.")

        query_vec = self.get_embedding(query)
        scored_results = []

        for entry in self.entries:
            score = cosine_similarity(query_vec, entry["vector"])
            scored_results.append({
                "chunk_id": entry["chunk_id"],
                "page_number": entry["page_number"],
                "source": entry["source"],
                "text": entry["text"],
                "similarity_score": round(score, 4)
            })

        # Sort descending by similarity score
        scored_results.sort(key=lambda x: x["similarity_score"], reverse=True)
        return scored_results[:top_k]


if __name__ == "__main__":
    # Self-test block: Embed a sample of chunks and run a test query
    print("=== Testing Vector Store & Semantic Search ===")

    # Locate cache directory
    script_dir = os.path.dirname(os.path.abspath(__file__)) if "__file__" in locals() else "."
    cache_path = os.path.join(script_dir, "vector_cache", "sample_embeddings.json")

    store = SimpleVectorStore()

    # Try loading existing cache first
    if not store.load_cache(cache_path):
        print("No cache found. Simulating with 3 distinct test chunks...")
        sample_chunks = [
            {
                "chunk_id": "chunk_academic_1",
                "page_number": 42,
                "source": "AASTU Handbook",
                "text": "Academic Standing: A student must maintain a minimum Cumulative Grade Point Average (CGPA) of 2.00 to remain in good standing. Failure to do so leads to academic probation."
            },
            {
                "chunk_id": "chunk_library_1",
                "page_number": 88,
                "source": "AASTU Handbook",
                "text": "Library Services: The university library opens at 8:00 AM and closes at 10:00 PM on weekdays. Valid student ID card is mandatory for entry."
            },
            {
                "chunk_id": "chunk_sports_1",
                "page_number": 115,
                "source": "AASTU Handbook",
                "text": "Campus Recreation: The university gymnasium and sports fields are open for all registered undergraduate students during scheduled athletic hours."
            }
        ]
        store.add_chunks(sample_chunks, batch_size=10)
        store.save_cache(cache_path)

    # Test Query: Student asking about minimum CGPA
    test_query = "What is the minimum CGPA required for good academic standing?"
    print(f"\nQuery: '{test_query}'")
    results = store.search(test_query, top_k=2)

    print("\n--- Search Results (Top Matches) ---")
    for rank, r in enumerate(results, 1):
        print(f"\n#{rank} [Score: {r['similarity_score']}] (Page {r['page_number']})")
        print(f"Excerpt: {r['text'][:140]}...")

=== Testing Vector Store & Semantic Search ===
No cache found. Simulating with 3 distinct test chunks...
Embedding 3 chunks in batches of 10...
  Processed 3/3 chunks...
[OK] Cached 3 vector entries to: ./vector_cache/sample_embeddings.json

Query: 'What is the minimum CGPA required for good academic standing?'

--- Search Results (Top Matches) ---

#1 [Score: 0.8275] (Page 42)
Excerpt: Academic Standing: A student must maintain a minimum Cumulative Grade Point Average (CGPA) of 2.00 to remain in good standing. Failure to do...

#2 [Score: 0.5659] (Page 115)
Excerpt: Campus Recreation: The university gymnasium and sports fields are open for all registered undergraduate students during scheduled athletic h...


In [5]:
import os

# Create the prompts directory in Colab
os.makedirs("/content/prompts", exist_ok=True)

prompt_text = """You are the official AASTU Student Academic Advisor AI.
Your job is to answer student questions accurately and professionally based ONLY on the excerpts provided below.

RULES:
1. Grounding: Answer strictly using facts from the provided context. If the answer is NOT present, state clearly: "I cannot find this information in the official AASTU handbook."
2. Citations: Every fact you state must cite the chunk_id and page_number from where it was retrieved.
3. Tone: Helpful, direct, and professional.
4. Output Format: You MUST reply in valid JSON adhering exactly to this JSON schema:

{
  "answer": "Clear, concise natural language answer to the student.",
  "citations": [
    {
      "chunk_id": "chunk_id_here",
      "page_number": 42,
      "source": "AASTU Handbook"
    }
  ],
  "confidence_score": 0.95,
  "is_grounded": true
}

---
CONTEXT EXCERPTS:
{context}

---
STUDENT QUESTION:
{question}
"""

with open("/content/prompts/system_prompt.txt", "w", encoding="utf-8") as f:
    f.write(prompt_text)

print("[OK] Created /content/prompts/system_prompt.txt successfully in Colab!")

[OK] Created /content/prompts/system_prompt.txt successfully in Colab!


In [8]:
"""
RAG Engine: Retrieve -> Augment -> Generate with Structured Pydantic Validation.
Loads system prompts from file, calls Gemini API, and validates outputs against
a strict Pydantic schema with automated repair retries.
"""

import os
import json
import urllib.request
import urllib.error
from typing import List, Dict, Any, Optional
from pydantic import BaseModel, Field, ValidationError

# Import vector store (handles local .py files AND interactive Colab cells)
try:
    from src.vector_store import SimpleVectorStore, get_gemini_api_key
except ImportError:
    try:
        from vector_store import SimpleVectorStore, get_gemini_api_key
    except ImportError:
        # In Google Colab, the class is already in memory from the previous cell!
        if "SimpleVectorStore" not in globals():
            raise ImportError(
                "SimpleVectorStore not found! Make sure you ran the vector_store cell above first."
            )


# ==========================================
# 1. Pydantic Models for Structured Output
# ==========================================
class Citation(BaseModel):
    chunk_id: str = Field(description="Unique ID of the source chunk")
    page_number: int = Field(description="Page number from the official handbook")
    source: str = Field(default="AASTU Handbook", description="Document source name")


class RAGResponse(BaseModel):
    answer: str = Field(description="Direct, factual answer to the question")
    citations: List[Citation] = Field(default_factory=list, description="List of source citations")
    confidence_score: float = Field(ge=0.0, le=1.0, description="Confidence score between 0.0 and 1.0")
    is_grounded: bool = Field(description="True if the answer is directly supported by evidence")


# ==========================================
# 2. Complete RAG Engine Class
# ==========================================
class RAGEngine:
    def __init__(self, vector_store: SimpleVectorStore, prompt_path: Optional[str] = None):
        self.vector_store = vector_store
        self.api_key = vector_store.api_key
        self.prompt_template = self._load_prompt_template(prompt_path)

    def _load_prompt_template(self, prompt_path: Optional[str]) -> str:
        """Loads prompt from file as required by Slide 13."""
        candidates = [
            prompt_path,
            os.path.join("prompts", "system_prompt.txt"),
            os.path.join("..", "prompts", "system_prompt.txt"),
            "/content/prompts/system_prompt.txt",
            "/content/system_prompt.txt",
        ]
        for p in candidates:
            if p and os.path.exists(p):
                with open(p, "r", encoding="utf-8") as f:
                    return f.read()

        # Fallback default if file is missing
        return (
            "You are an academic advisor. Answer using ONLY the context.\n"
            "Output JSON: {\"answer\": string, \"citations\": [{\"chunk_id\": string, \"page_number\": int}], "
            "\"confidence_score\": float, \"is_grounded\": bool}\n\n"
            "CONTEXT:\n{context}\n\nQUESTION:\n{question}"
        )

    def _call_gemini(self, prompt: str) -> str:
        """Calls Gemini API requesting JSON structured output."""
        url = (
            f"https://generativelanguage.googleapis.com/v1beta/models/"
            f"gemini-flash-lite-latest:generateContent?key={self.api_key}"
        )
        payload = {
            "contents": [{"parts": [{"text": prompt}]}],
            "generationConfig": {
                "responseMimeType": "application/json",
                "temperature": 0.2  # Low temperature for factual, deterministic answers
            }
        }
        data = json.dumps(payload).encode("utf-8")
        req = urllib.request.Request(url, data=data, headers={"Content-Type": "application/json"})

        with urllib.request.urlopen(req) as resp:
            raw_response = json.loads(resp.read().decode("utf-8"))
            return raw_response["candidates"][0]["content"]["parts"][0]["text"].strip()

    def query(self, question: str, top_k: int = 3, max_repair_attempts: int = 2) -> RAGResponse:
        """
        Executes the full RAG pipeline:
        Retrieve -> Augment -> Generate -> Pydantic Validation & Repair Loop.
        """
        # Step 1: RETRIEVE
        retrieved_chunks = self.vector_store.search(question, top_k=top_k)

        # Step 2: AUGMENT
        context_parts = []
        for c in retrieved_chunks:
            context_parts.append(
                f"[Chunk: {c['chunk_id']} | Page: {c['page_number']}]\n{c['text']}"
            )
        context_str = "\n\n".join(context_parts)

        # Use .replace() so it doesn't conflict with JSON braces in the prompt
        full_prompt = (
            self.prompt_template
            .replace("{context}", context_str)
            .replace("{question}", question)
        )

        # Step 3 & 4: GENERATE with VALIDATION & REPAIR LOOP (Slide 6 & 13)
        current_prompt = full_prompt
        for attempt in range(max_repair_attempts + 1):
            raw_json = self._call_gemini(current_prompt)
            try:
                # Validate with Pydantic
                validated_response = RAGResponse.model_validate_json(raw_json)
                return validated_response
            except ValidationError as err:
                if attempt < max_repair_attempts:
                    print(f"[Repair Loop] Validation failed on attempt {attempt + 1}. Retrying with error...")
                    current_prompt = (
                        f"{full_prompt}\n\n"
                        f"CRITICAL ERROR: Your previous response violated the schema:\n{err}\n"
                        f"Please fix and output ONLY valid JSON."
                    )
                else:
                    raise RuntimeError(f"RAG output failed validation after {max_repair_attempts} retries: {err}")


if __name__ == "__main__":
    print("=== Testing Full RAG Engine ===")

    # 1. Initialize Vector Store
    store = SimpleVectorStore()

    # 2. Add sample chunks if empty
    if not store.entries:
        sample_chunks = [
            {
                "chunk_id": "chunk_policy_1",
                "page_number": 42,
                "source": "AASTU Handbook",
                "text": "Academic Standing: A student must maintain a minimum Cumulative Grade Point Average (CGPA) of 2.00 to remain in good standing. A student whose CGPA falls below 2.00 is placed on academic probation."
            },
            {
                "chunk_id": "chunk_policy_2",
                "page_number": 45,
                "source": "AASTU Handbook",
                "text": "Dismissal and Appeals: A student subject to academic dismissal may submit an appeal letter to the Academic Standards Committee within 10 working days of grade announcement."
            }
        ]
        store.add_chunks(sample_chunks)

    # 3. Create RAG Engine
    engine = RAGEngine(vector_store=store)

    # 4. Test question
    question = "What is the minimum CGPA for academic standing, and can I appeal if dismissed?"
    print(f"\nStudent Question: {question}\n")

    response = engine.query(question, top_k=2)

    print("--- Validated Structured RAG Response ---")
    print(f"Answer: {response.answer}\n")
    print(f"Confidence: {response.confidence_score * 100}% | Grounded: {response.is_grounded}")
    print("Citations:")
    for c in response.citations:
        print(f"  * Source: {c.source} | Page: {c.page_number} | Chunk: {c.chunk_id}")

=== Testing Full RAG Engine ===
Embedding 2 chunks in batches of 20...
  Processed 2/2 chunks...

Student Question: What is the minimum CGPA for academic standing, and can I appeal if dismissed?

--- Validated Structured RAG Response ---
Answer: To remain in good standing, you must maintain a minimum Cumulative Grade Point Average (CGPA) of 2.00. If you are subject to academic dismissal, you can submit an appeal letter to the Academic Standards Committee within 10 working days of the grade announcement.

Confidence: 100.0% | Grounded: True
Citations:
  * Source: AASTU Handbook | Page: 42 | Chunk: chunk_policy_1
  * Source: AASTU Handbook | Page: 45 | Chunk: chunk_policy_2


In [9]:
import os

os.makedirs("/content/eval", exist_ok=True)

golden_data = """[
  {
    "test_id": "test_01_gpa_requirement",
    "question": "What is the minimum Cumulative Grade Point Average (CGPA) required for good academic standing?",
    "expected_keywords": ["2.00", "standing"],
    "expected_page": 42,
    "should_be_grounded": true,
    "description": "Tests if RAG retrieves and cites the correct minimum GPA."
  },
  {
    "test_id": "test_02_dismissal_appeal",
    "question": "How many days do I have to submit an appeal after being subject to academic dismissal?",
    "expected_keywords": ["10", "working days", "appeal"],
    "expected_page": 45,
    "should_be_grounded": true,
    "description": "Tests numerical timeline extraction for appeals."
  },
  {
    "test_id": "test_03_library_hours",
    "question": "What are the opening and closing hours for the university library on weekdays?",
    "expected_keywords": ["8:00 AM", "10:00 PM"],
    "expected_page": 88,
    "should_be_grounded": true,
    "description": "Tests retrieval of operational facility information."
  },
  {
    "test_id": "test_04_out_of_scope_hallucination_check",
    "question": "What is the secret recipe for the campus cafeteria's special burger sauce?",
    "expected_keywords": ["cannot find", "not"],
    "expected_page": null,
    "should_be_grounded": false,
    "description": "Adversarial test: Verifies that the AI gracefully refuses to hallucinate when data is missing."
  }
]"""

with open("/content/eval/golden_dataset.json", "w", encoding="utf-8") as f:
    f.write(golden_data)

print("[OK] Created /content/eval/golden_dataset.json successfully!")

[OK] Created /content/eval/golden_dataset.json successfully!


In [10]:
"""
Automated Evaluation Suite for AASTU RAG System.
Runs regression tests against a golden dataset to evaluate retrieval hit-rate,
factual grounding, schema validity, and hallucination resistance.
"""

import os
import sys
import json
import time
from typing import Dict, Any, List

# Add parent directory to path so we can import from src/
current_dir = os.path.dirname(os.path.abspath(__file__)) if "__file__" in locals() else "."
project_root = os.path.dirname(current_dir) if os.path.basename(current_dir) == "eval" else current_dir
sys.path.insert(0, project_root)

# Import RAG engine and vector store (works both locally and in Colab)
try:
    from src.rag_engine import RAGEngine, RAGResponse
    from src.vector_store import SimpleVectorStore
except ImportError:
    try:
        from rag_engine import RAGEngine, RAGResponse
        from vector_store import SimpleVectorStore
    except ImportError:
        # Fallback for interactive Colab memory
        if "RAGEngine" not in globals() or "SimpleVectorStore" not in globals():
            raise ImportError("Please run the vector_store and rag_engine cells first!")


def load_golden_dataset() -> List[Dict[str, Any]]:
    """Loads the test cases from disk."""
    candidates = [
        os.path.join("eval", "golden_dataset.json"),
        os.path.join(project_root, "eval", "golden_dataset.json"),
        "/content/eval/golden_dataset.json",
        "/content/golden_dataset.json",
        "golden_dataset.json",
    ]
    for p in candidates:
        if os.path.exists(p):
            with open(p, "r", encoding="utf-8") as f:
                return json.load(f)
    raise FileNotFoundError("Could not find 'golden_dataset.json' in eval/ or current directory.")


def seed_test_knowledge_base(store: SimpleVectorStore) -> None:
    """Seeds the vector store with reference chunks to ensure repeatable evals."""
    reference_chunks = [
        {
            "chunk_id": "chunk_policy_1",
            "page_number": 42,
            "source": "AASTU Handbook",
            "text": "Academic Standing: A student must maintain a minimum Cumulative Grade Point Average (CGPA) of 2.00 to remain in good standing. Failure to do so leads to academic probation."
        },
        {
            "chunk_id": "chunk_policy_2",
            "page_number": 45,
            "source": "AASTU Handbook",
            "text": "Dismissal and Appeals: A student subject to academic dismissal may submit an appeal letter to the Academic Standards Committee within 10 working days of grade announcement."
        },
        {
            "chunk_id": "chunk_library_1",
            "page_number": 88,
            "source": "AASTU Handbook",
            "text": "Library Services: The university library opens at 8:00 AM and closes at 10:00 PM on weekdays. Valid student ID card is mandatory for entry."
        },
        {
            "chunk_id": "chunk_sports_1",
            "page_number": 115,
            "source": "AASTU Handbook",
            "text": "Campus Recreation: The university gymnasium and sports fields are open for all registered undergraduate students during scheduled athletic hours."
        }
    ]
    store.add_chunks(reference_chunks, batch_size=10)


def run_evaluation() -> None:
    print("=" * 65)
    print("      AASTU RAG PIPELINE - AUTOMATED EVALUATION SUITE")
    print("=" * 65)

    test_cases = load_golden_dataset()
    print(f"[OK] Loaded {len(test_cases)} golden test cases.\n")

    # 1. Initialize RAG system
    store = SimpleVectorStore()
    seed_test_knowledge_base(store)
    engine = RAGEngine(vector_store=store)

    results = []
    retrieval_hits = 0
    grounding_passes = 0
    schema_passes = 0

    print("Running test cases...\n")

    for idx, test in enumerate(test_cases, 1):
        t_id = test["test_id"]
        question = test["question"]
        expected_keywords = test["expected_keywords"]
        expected_page = test["expected_page"]
        should_ground = test["should_be_grounded"]

        print(f"[{idx}/{len(test_cases)}] Running: {t_id}")
        print(f"    Question: \"{question}\"")

        try:
            # Query the RAG engine
            response: RAGResponse = engine.query(question, top_k=2)
            schema_passes += 1

            # Check 1: Retrieval Hit Check
            retrieved_pages = [c.page_number for c in response.citations]
            hit = (expected_page in retrieved_pages) if expected_page else True
            if hit:
                retrieval_hits += 1

            # Check 2: Keyword / Grounding Check
            answer_lower = response.answer.lower()
            keywords_found = [kw for kw in expected_keywords if kw.lower() in answer_lower]
            keyword_pass = len(keywords_found) >= 1

            # Check 3: Hallucination check on out-of-scope question
            if not should_ground:
                # For out-of-scope, passing means it refused or flagged low confidence
                hallucination_safe = (
                    "cannot find" in answer_lower or
                    "not" in answer_lower or
                    not response.is_grounded or
                    response.confidence_score < 0.6
                )
                test_passed = hallucination_safe
                status = "PASS (Safely Refused)" if test_passed else "FAIL (Hallucinated)"
            else:
                test_passed = hit and keyword_pass and response.is_grounded
                status = "PASS" if test_passed else "FAIL"

            if test_passed:
                grounding_passes += 1

            print(f"    Status: {status}")
            print(f"    Answer: \"{response.answer[:100]}...\"")
            print(f"    Citations: {retrieved_pages} | Expected Page: {expected_page}")
            print(f"    Keywords Found: {keywords_found}/{expected_keywords}\n")

            results.append({
                "test_id": t_id,
                "status": status,
                "passed": test_passed
            })

            time.sleep(1.0)  # Gentle spacing for API rate limits

        except Exception as e:
            print(f"    Status: ERROR ({e})\n")
            results.append({"test_id": t_id, "status": f"ERROR: {e}", "passed": False})

    # ==========================================
    # Final Scorecard Summary (Slide 10 & 14)
    # ==========================================
    total = len(test_cases)
    total_passed = sum(1 for r in results if r["passed"])
    accuracy = (total_passed / total) * 100
    hit_rate = (retrieval_hits / total) * 100

    print("=" * 65)
    print("                    EVALUATION SCORECARD")
    print("=" * 65)
    print(f"  Total Test Cases       : {total}")
    print(f"  Passed                 : {total_passed} / {total}")
    print(f"  Overall Accuracy       : {accuracy:.1f}%")
    print(f"  Schema Conformance     : {(schema_passes / total) * 100:.1f}%")
    print(f"  Retrieval Hit-Rate     : {hit_rate:.1f}%")
    print("=" * 65)

    if accuracy >= 75.0:
        print("[SUCCESS] Quality bar passed! Ready for production deployment.")
    else:
        print("[WARNING] Quality bar not met. Refine prompts or chunking.")
    print("=" * 65)


if __name__ == "__main__":
    run_evaluation()

      AASTU RAG PIPELINE - AUTOMATED EVALUATION SUITE
[OK] Loaded 4 golden test cases.

Embedding 4 chunks in batches of 10...
  Processed 4/4 chunks...
Running test cases...

[1/4] Running: test_01_gpa_requirement
    Question: "What is the minimum Cumulative Grade Point Average (CGPA) required for good academic standing?"
    Status: PASS
    Answer: "To remain in good academic standing, a student must maintain a minimum Cumulative Grade Point Averag..."
    Citations: [42] | Expected Page: 42
    Keywords Found: ['2.00', 'standing']/['2.00', 'standing']

[2/4] Running: test_02_dismissal_appeal
    Question: "How many days do I have to submit an appeal after being subject to academic dismissal?"
    Status: PASS
    Answer: "You have 10 working days from the date of grade announcement to submit an appeal letter to the Acade..."
    Citations: [45] | Expected Page: 45
    Keywords Found: ['10', 'working days', 'appeal']/['10', 'working days', 'appeal']

[3/4] Running: test_03_library_